## Monads

https://cs3110.github.io/textbook/chapters/ds/monads.html

A monad is more of a design pattern than a data structure.

The name “monad” comes from the mathematical field of category theory, which studies abstractions of mathematical structures.

Monads became popular in the programming world through their use in Haskell, a functional programming language that is even more pure than OCaml—that is, Haskell avoids side effects and imperative features even more than OCaml.

Monads are used to model computations. Think of a computation as being like a function, which maps an input to an output, but as also doing “something more.” The something more is an effect that the function has as a result of being computed. For example, the effect might involve printing to the screen. Monads provide an abstraction of effects, and help to make sure that effects happen in a controlled order.

Example monads:
* Maybe Monad (option, result)
* Writer Monad (showList, ShowExp.show)
* The Lwt Monad (promise)

#### Category theory link:
A Sensible Introduction to Category Theory
https://www.youtube.com/watch?v=yAi3XWCBkDo
(Monads, Functors)

In [4]:
let ( / ) = Stdlib.( / )
(* works fine for 2 *)
let x = 1 + (4 / 2)

(* PROBLEM: exception with divided by 0 *)

val ( / ) : int -> int -> int = <fun>


error: runtime_error

In [6]:
(* Loss of compositionality when some operators have to deal with exceptions or partial outputs. *)
let div (x:int) (y:int) : int option =
  if y = 0 then None else Some (x / y)

let ( / ) = div

(* PROBLEM: won't type check *)
let x = 1 + (4 / 2)

error: compile_error

In [7]:
(* A fix with a tremendous amount of code duplication. *)

let plus_opt (x:int option) (y:int option) : int option =
  match x,y with
  | None, _ | _, None -> None
  | Some a, Some b -> Some (Stdlib.( + ) a b)

let ( + ) = plus_opt

let minus_opt (x:int option) (y:int option) : int option =
  match x,y with
  | None, _ | _, None -> None
  | Some a, Some b -> Some (Stdlib.( - ) a b)

let ( - ) = minus_opt

let mult_opt (x:int option) (y:int option) : int option =
  match x,y with
  | None, _ | _, None -> None
  | Some a, Some b -> Some (Stdlib.( * ) a b)

let ( * ) = mult_opt

let div_opt (x:int option) (y:int option) : int option =
  match x,y with
  | None, _ | _, None -> None
  | Some a, Some b ->
    if b=0 then None else Some (Stdlib.( / ) a b)

let ( / ) = div_opt

val plus_opt : int option -> int option -> int option = <fun>


val ( + ) : int option -> int option -> int option = <fun>


val minus_opt : int option -> int option -> int option = <fun>


val ( - ) : int option -> int option -> int option = <fun>


val mult_opt : int option -> int option -> int option = <fun>


val ( * ) : int option -> int option -> int option = <fun>


val div_opt : int option -> int option -> int option = <fun>


val ( / ) : int option -> int option -> int option = <fun>


In [8]:
(* does type check *)

let x = Some 1 + (Some 4 / Some 2)

val x : int option = Some 3


In [10]:
(* Let's consolidate the duplications. *)

let propagate_none (op : int -> int -> int) (x : int option) (y : int option) =
  match x, y with
  | None, _ | _, None -> None
  | Some a, Some b -> Some (op a b)

let ( + ) = propagate_none Stdlib.( + )
let ( - ) = propagate_none Stdlib.( - )
let ( * ) = propagate_none Stdlib.( * )

(* This is a programming trick using software engineering techniques to 
   reduce engineering effort and improve code clarity, 
   but it does not improve math/mental clarity of the computation *)

val propagate_none :
  (int -> int -> int) -> int option -> int option -> int option = <fun>


val ( + ) : int option -> int option -> int option = <fun>


val ( - ) : int option -> int option -> int option = <fun>


val ( * ) : int option -> int option -> int option = <fun>


In [9]:
(* Operator "/" is harder to consolidate because its return type is already int option. *)

let propagate_none
  (op : int -> int -> int option) (x : int option) (y : int option)
=
  match x, y with
  | None, _ | _, None -> None
  | Some a, Some b -> op a b

let ( / ) = propagate_none div

val propagate_none :
  (int -> int -> int option) -> int option -> int option -> int option =
  <fun>


val ( / ) : int option -> int option -> int option = <fun>


In [10]:
(* Make the other three work with the new propagate_none. *)

let wrap_output (op : int -> int -> int) (x : int) (y : int) : int option =
  Some (op x y)

let ( + ) = propagate_none (wrap_output Stdlib.( + ))
let ( - ) = propagate_none (wrap_output Stdlib.( - ))
let ( * ) = propagate_none (wrap_output Stdlib.( * ))

val wrap_output : (int -> int -> int) -> int -> int -> int option = <fun>


val ( + ) : int option -> int option -> int option = <fun>


val ( - ) : int option -> int option -> int option = <fun>


val ( * ) : int option -> int option -> int option = <fun>


In [11]:
let div (x : int) (y : int) : int option =
  if y = 0 then None else wrap_output Stdlib.( / ) x y

let ( / ) = propagate_none div

val div : int -> int -> int option = <fun>


val ( / ) : int option -> int option -> int option = <fun>


In [12]:
(* The simple int-based expression still doesn't work. *)
let x = 1 + (4 / 2)

error: compile_error

In [13]:
(* We finally make this work, without duplidated code and with a higher-level abstraction. *)

let x = Some 1 + (Some 4 / Some 2)

val x : int option = Some 3


### The OCaml version of the "Maybe Monad" with Return and Bind

Now let's capture the fundamental idea in the previous example with two Monad operations: return and bind.

In [14]:
let return (x : int) : int option = Some x

val return : int -> int option = <fun>


In [16]:
(* Bind takes two parameters: x and a function op. Function op is applied only when x is not None. *)

let bind (x : int option) (op : int -> int option) : int option =
  match x with
  | None -> None
  | Some a -> op a

let ( >>= ) = bind

val bind : int option -> (int -> int option) -> int option = <fun>


val ( >>= ) : int option -> (int -> int option) -> int option = <fun>


In [17]:
let upgrade : (int -> int option) -> (int option -> int option) =
  fun (op : int -> int option) (x : int option) -> (x >>= op)

val upgrade : (int -> int option) -> int option -> int option = <fun>


In [38]:
let upgrade op x = x >>= op

val upgrade : (int -> int option) -> int option -> int option = <fun>


In [18]:
let ( + ) (x : int option) (y : int option) : int option =
  x >>= fun a ->
  y >>= fun b ->
  return (Stdlib.( + ) a b)

let ( - ) (x : int option) (y : int option) : int option =
  x >>= fun a ->
  y >>= fun b ->
  return (Stdlib.( - ) a b)

let ( * ) (x : int option) (y : int option) : int option =
  x >>= fun a ->
  y >>= fun b ->
  return (Stdlib.( * ) a b)

let ( / ) (x : int option) (y : int option) : int option =
  x >>= fun a ->
  y >>= fun b ->
  if b = 0 then None else return (Stdlib.( / ) a b)

val ( + ) : int option -> int option -> int option = <fun>


val ( - ) : int option -> int option -> int option = <fun>


val ( * ) : int option -> int option -> int option = <fun>


val ( / ) : int option -> int option -> int option = <fun>


In [19]:
let x = 1 + (4 / 2)

error: compile_error

In [20]:
let x = Some 1 + (Some 4 / Some 2)

val x : int option = Some 3


In [18]:
let upgrade_binary op x y =
  x >>= fun a ->
  y >>= fun b ->
  op a b

let return_binary op x y = return (op x y)

let ( + ) = upgrade_binary (return_binary Stdlib.( + ))
let ( - ) = upgrade_binary (return_binary Stdlib.( - ))
let ( * ) = upgrade_binary (return_binary Stdlib.( * ))
let ( / ) = upgrade_binary div

error: compile_error

In [19]:
let x = Some 1 + ( Some 4 / Some 2)

val x : int option = Some 3


### Monads in OCaml

The concept of monads in CS started in Haskell. In OCaml, a monad is a structure that meets the following properties. 

Monads are used to control side effects, e.g. printing to the screen is a side effect. 

In [44]:
module type Monad = sig
  type 'a t
  val return : 'a -> 'a t
  val bind : 'a t -> ('a -> 'b t) -> 'b t
end



module type Monad =
  sig
    type 'a t
    val return : 'a -> 'a t
    val bind : 'a t -> ('a -> 'b t) -> 'b t
  end


In [1]:
(* Or, identically: *)
module type Monad = sig
  type 'a t
  val return : 'a -> 'a t
  val ( >>= ) : 'a t -> ('a -> 'b t) -> 'b t
end

module type Monad =
  sig
    type 'a t
    val return : 'a -> 'a t
    val ( >>= ) : 'a t -> ('a -> 'b t) -> 'b t
  end


In [47]:
module Maybe : Monad = struct
  type 'a t = 'a option

  let return x = Some x

  let (>>=) m f =
    match m with
    | None -> None
    | Some x -> f x
    
  let bind = (>>=)
end

module Maybe : Monad


Monads are used to model computations.  A computation is like a function, which maps an input to an output, but as also doing “something more”, a.k.a. side effects, which are a result of the computation. For example, the effect might involve printing to the screen. Monads provide an abstraction of effects, and help make sure that effects happen in a controlled order.

In the "Maybe Monad", we can think of these “upgraded” functions as computations that may have the effect of producing nothing. 

### Challenge: How to write a Print Monad that has the side effect of printing the AST of the EXPRs

Hint: The Writer Monad in the textbook is a Print Monad that prints specific log information.

### Challenge: Write and test the Lwt Monad. 

Hint: it's trivial because Lwt was already implemented as a monad, although it didn't explicitly use the names "return" and "bind".

### Challenge: Data structure with exceptions

Remember the different ways of dealing an empty stack when popping elements from it? How to write a monad for it?

Similarly, what to do when the function find results in Not_found for other data structures.

### The Three Monad Laws

Law 1: Having the trivial effect on a value, then binding a function on it, is the same as just calling the function on the value. 

I.e. `Law 1: The monads do not alter the nature of the computation.`

Law 2: Binding on the trivial effect is the same as just not having the effect. 

I.e. `Law 2: The return operation of monads do not alter the result of the computation.`

Law 3: The binding operation is associative.


#### In short, here are the three monad laws.

Law 1: `return x >>= f` behaves the same as `f x`.

Law 2: `m >>= return` behaves the same as `m`.

Law 3: `(m >>= f) >>= g` behaves the same as `m >>= (fun x -> f x >>= g)`.

With the `compose` operator, the three laws can be further simplied to:

Law 1: `return >=> f` behaves the same as `f`.

Law 2: `f >=> return` behaves the same as `f`.

Law 3: `(f >=> g) >=> h` behaves the same as `f >=> (g >=> h)`.

In [21]:
let compose f g x =
  f x >>= fun y ->
  g y

let ( >=> ) = compose

let ( >>= ) = bind

val ( >>= ) : int option -> (int -> int option) -> int option = <fun>
